# ⚡ Optimized Historical Weather Data Extraction
### Key optimizations applied:
- **Vectorized interpolation**: All 300 cities interpolated in ONE call per variable (vs 300 calls)
- **GRIB pre-loading**: `.load()` into RAM before city loop — eliminates repeated disk I/O
- **Parallel blob processing**: ThreadPoolExecutor downloads/processes multiple tar.gz files concurrently
- **Batched DB writes**: Single `push_data` call per blob (vs 300 per blob)
- **Checkpointing**: Skips already-processed blobs automatically
- **cfgrib index caching**: Reuses `.idx` files to avoid re-scanning GRIB on reruns

In [ ]:
# Install dependencies (run once)
!pip install xarray cfgrib eccodes
%restart_python

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import os
import tarfile
import logging
import time
import traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from warnings import filterwarnings

import cfgrib
import xarray as xr
import numpy as np
import pandas as pd
from google.cloud import storage

filterwarnings('ignore')

sys.path.append('/Workspace/Power Pricing/PowerDE_Data/')
from functions_UDF1 import fetch_table_in_pandas, push_data

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger(__name__)
log.info('Imports complete')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
BUCKET_NAME      = 'gcp10-sb-shortterm-forecasting'
FOLDER_PATH      = 'nwp/ncmrwf-0p125-re/'
DOWNLOAD_DIR     = Path('data_downloads')
CHECKPOINT_FILE  = Path('processed_blobs.txt')   # tracks completed blobs
YEAR_FILTER      = '2023'                         # change or remove as needed
TARGET_DS_INDICES = (1, 2, 3, 6)                  # dataset indices inside each GRIB
MAX_WORKERS      = 4                              # parallel blob download/process threads
                                                  # tune to GCS bandwidth & cluster cores

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# ── Variable name mapping ─────────────────────────────────────────────────────
VARIABLES_MAPPING = {
    'v'           : 'northward_wind_50mt',
    'u'           : 'eastward_wind_50mt',
    'u10'         : 'eastward_wind_10mt',
    'v10'         : 'northward_wind_10mt',
    't2m'         : 'air_temperature_2mt',
    'sh2'         : 'specific_humidity_2mt',
    'gh'          : 'gropotential_height',
    'sp'          : 'surface_pressure',
    'prmsl'       : 'pressure_reduce_to_mean_sea_level',
    'avg_sdswrf'  : 'solar_radiation',
    'unknown'     : 'rainfall',
}

log.info('Config ready')

In [ ]:
# ── Load city coordinates ─────────────────────────────────────────────────────
latlong_df = fetch_table_in_pandas('dim_latlong')

# Precompute coordinate arrays — used for vectorised interpolation
city_lats = xr.DataArray(latlong_df['lat'].astype(float).values, dims='city')
city_lons = xr.DataArray(latlong_df['long'].astype(float).values, dims='city')

log.info(f'Loaded {len(latlong_df)} cities')
display(latlong_df.head())

In [ ]:
# ── GCS client ───────────────────────────────────────────────────────────────
storage_client = storage.Client.from_service_account_json(
    'agel-svc-shorttermf-qa-storage_bucket_view.json'
)
bucket = storage_client.bucket(BUCKET_NAME)
log.info('GCS client ready')

In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────
def load_checkpoint() -> set:
    """Return set of already-processed blob names."""
    if CHECKPOINT_FILE.exists():
        return set(CHECKPOINT_FILE.read_text().splitlines())
    return set()

def save_checkpoint(blob_name: str):
    """Append a completed blob name to the checkpoint file."""
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(blob_name + '\n')

PROCESSED_BLOBS = load_checkpoint()
log.info(f'Already processed: {len(PROCESSED_BLOBS)} blobs')

In [ ]:
# ── Core helper: download + extract tar.gz ────────────────────────────────────
def download_and_extract_grib(blob_obj) -> str:
    """
    Download a GCS blob (tar.gz) and extract it locally.
    Returns the path of the extracted .grib2 file.
    """
    tar_filename = blob_obj.name.split('/')[-1]
    tar_path     = DOWNLOAD_DIR / tar_filename

    log.info(f'Downloading {tar_filename} ...')
    t0 = time.time()
    blob_obj.download_to_filename(str(tar_path))
    log.info(f'Downloaded {tar_filename} in {time.time()-t0:.1f}s')

    with tarfile.open(tar_path, 'r:gz') as tar:
        members = tar.getmembers()
        tar.extractall(path=str(DOWNLOAD_DIR))

    # Clean up tar to save disk space
    tar_path.unlink(missing_ok=True)

    grib_member = members[-1].name   # last member is the .grib2 file
    log.info(f'Extracted: {grib_member}')
    return grib_member


# ── Core helper: vectorised extraction for ALL cities from one dataset ────────
def extract_all_cities_from_dataset(ds: xr.Dataset) -> pd.DataFrame:
    """
    CRITICAL OPTIMISATION:
    Instead of calling ds[var].interp(lat=lat, lon=lon) 300 times,
    we call it ONCE with arrays of all 300 lat/lon values.
    xarray broadcasts this into a (time, step, city) array in a single pass.
    """
    frames = []

    for var in ds.data_vars.keys():
        if var not in VARIABLES_MAPPING:
            continue

        log.info(f'  Vectorised interpolation: {var} -> {VARIABLES_MAPPING[var]}')

        # Single interp call for ALL cities — shape: (time, step, city)
        interpolated = ds[var].interp(
            latitude=city_lats,
            longitude=city_lons,
            method='linear'
        )

        # Stack time dimensions and convert to DataFrame
        df = interpolated.to_dataframe(name=VARIABLES_MAPPING[var]).reset_index()

        # Keep only the columns we need
        keep = [c for c in ['time', 'valid_time', 'city', VARIABLES_MAPPING[var]] if c in df.columns]
        df = df[keep].rename(columns={'time': 'source_timestamp', 'valid_time': 'forecast_timestamp'})
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    # Merge all variables on (source_timestamp, forecast_timestamp, city)
    result = frames[0]
    for df in frames[1:]:
        result = result.merge(
            df,
            on=['source_timestamp', 'forecast_timestamp', 'city'],
            how='outer'
        )
    return result


# ── Core helper: process one GRIB file for all cities ─────────────────────────
def process_grib_file(grib_path: str) -> pd.DataFrame:
    """
    Open GRIB, pre-load required datasets into RAM, run vectorised
    interpolation for all cities, return a combined DataFrame.
    """
    log.info(f'Opening GRIB: {grib_path}')
    t0 = time.time()

    # indexpath=None reuses cached .idx files from prior runs (big speedup on reruns)
    datasets = cfgrib.open_datasets(
        grib_path,
        backend_kwargs={'indexing': {'indexpath': grib_path + '.{short_hash}.idx'}}
    )
    log.info(f'GRIB opened ({len(datasets)} datasets) in {time.time()-t0:.1f}s')

    all_ds_frames = []

    for i, ds in enumerate(datasets):
        if i not in TARGET_DS_INDICES:
            continue

        log.info(f'  Loading dataset index {i} into RAM ...')
        t1 = time.time()
        ds = ds.load()   # ← Pull all data from disk into memory ONCE
        log.info(f'  Loaded in {time.time()-t1:.1f}s | vars: {list(ds.data_vars.keys())}')

        df = extract_all_cities_from_dataset(ds)
        if not df.empty:
            all_ds_frames.append(df)

        ds.close()   # free RAM after each dataset

    if not all_ds_frames:
        return pd.DataFrame()

    # Combine across dataset indices
    combined = all_ds_frames[0]
    for df in all_ds_frames[1:]:
        combined = combined.merge(
            df,
            on=['source_timestamp', 'forecast_timestamp', 'city'],
            how='outer',
            suffixes=('', '_dup')
        )
    # Drop any accidental duplicate columns
    combined = combined[[c for c in combined.columns if not c.endswith('_dup')]]
    return combined


# ── Core helper: process one blob end-to-end ──────────────────────────────────
def process_blob(blob_obj) -> str:
    """
    Full pipeline for one GCS blob:
      download → extract → process GRIB → attach city metadata → push to DB
    Returns blob name on success, raises on failure.
    """
    blob_name = blob_obj.name
    tar_filename = blob_name.split('/')[-1]
    t_start = time.time()

    try:
        # ── Handle special pre-existing file ──────────────────────────────────
        if '20230611_00Z.tar.gz' in blob_name:
            grib_file = str(DOWNLOAD_DIR / 'fcst_20230611.grib2')
        else:
            grib_member = download_and_extract_grib(blob_obj)
            grib_file = str(DOWNLOAD_DIR / grib_member)

        # ── Extract data for ALL cities at once ────────────────────────────────
        result_df = process_grib_file(grib_file)

        if result_df.empty:
            log.warning(f'No data extracted from {blob_name}')
            return blob_name

        # ── Attach City / State from the 'city' index ──────────────────────────
        city_meta = latlong_df.reset_index()[['City', 'State']].copy()
        city_meta.index.name = 'city'
        city_meta = city_meta.reset_index()
        result_df = result_df.merge(city_meta, on='city', how='left').drop(columns='city')

        # ── Single batch push to DB ────────────────────────────────────────────
        log.info(f'Pushing {len(result_df):,} rows for {blob_name} ...')
        push_data(result_df, 'european_weather_forecast', 'append')

        # ── Clean up extracted grib to save disk space ─────────────────────────
        try:
            Path(grib_file).unlink(missing_ok=True)
            for idx_file in Path(DOWNLOAD_DIR).glob('*.idx'):
                idx_file.unlink(missing_ok=True)
        except Exception:
            pass

        elapsed = time.time() - t_start
        log.info(f'✅ Done: {blob_name} | {len(result_df):,} rows | {elapsed:.1f}s')
        return blob_name

    except Exception as e:
        log.error(f'❌ FAILED: {blob_name}\n{traceback.format_exc()}')
        raise


log.info('All helper functions defined ✓')

In [ ]:
# ── Main pipeline: parallel blob processing ───────────────────────────────────

log.info('Listing GCS blobs ...')
all_blobs = [
    b for b in bucket.list_blobs(prefix=FOLDER_PATH)
    if YEAR_FILTER in b.name and b.name.endswith('.tar.gz')
]

# Filter out already-processed blobs (checkpoint)
pending_blobs = [b for b in all_blobs if b.name not in PROCESSED_BLOBS]

log.info(f'Total blobs matching filter : {len(all_blobs)}')
log.info(f'Already processed (skipped) : {len(all_blobs) - len(pending_blobs)}')
log.info(f'Blobs to process now        : {len(pending_blobs)}')
log.info(f'Parallel workers            : {MAX_WORKERS}')

In [ ]:
# ── Execute parallel processing ───────────────────────────────────────────────
success_count = 0
fail_count    = 0
failed_blobs  = []

t_total = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_blob = {executor.submit(process_blob, b): b for b in pending_blobs}

    for future in as_completed(future_to_blob):
        blob_obj = future_to_blob[future]
        try:
            blob_name = future.result()
            save_checkpoint(blob_name)
            success_count += 1
            log.info(f'Progress: {success_count + fail_count}/{len(pending_blobs)} '
                     f'(✅ {success_count}  ❌ {fail_count})')
        except Exception:
            fail_count += 1
            failed_blobs.append(blob_obj.name)

total_elapsed = time.time() - t_total
log.info('=' * 60)
log.info(f'FINISHED  |  Total time: {total_elapsed/3600:.2f} hrs')
log.info(f'Success: {success_count}  |  Failed: {fail_count}')
if failed_blobs:
    log.warning(f'Failed blobs:\n' + '\n'.join(failed_blobs))

In [ ]:
# ── (Optional) Retry failed blobs sequentially ────────────────────────────────
# Useful when failures were due to transient GCS errors

retry_blobs = [b for b in all_blobs if b.name in failed_blobs]

for blob_obj in retry_blobs:
    log.info(f'Retrying: {blob_obj.name}')
    try:
        blob_name = process_blob(blob_obj)
        save_checkpoint(blob_name)
        log.info(f'✅ Retry success: {blob_name}')
    except Exception as e:
        log.error(f'❌ Retry also failed: {blob_obj.name} — {e}')